# Batch Normalization vs Layer Normalization

**Tech stack:** Python, TensorFlow (Keras), NumPy

This notebook builds a small dataset, applies Batch Normalization and Layer
Normalization to it using `tf.keras.layers`, and compares the results.

In [1]:
import numpy as np
import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)

I0000 00:00:1784997196.167201     572 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784997196.167668     572 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1784997196.215859     572 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1784997197.445964     572 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784997197.446275     572 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## Step 1: Create a small dataset

In [2]:
# 4 samples, 5 features each.
# Features are deliberately on very different scales (e.g. one column is
# ~1000, another is ~0.5) so the normalizing effect is easy to see.
data = np.array([
    [ 10.0, 200.0,  -5.0,  1000.0,  0.5],
    [ 12.0, 180.0,  -3.0,   950.0,  0.7],
    [  8.0, 220.0,  -7.0,  1050.0,  0.3],
    [ 11.0, 190.0,  -4.0,   980.0,  0.6],
], dtype=np.float32)

print("Original Input:")
print(data)

Original Input:
[[ 1.00e+01  2.00e+02 -5.00e+00  1.00e+03  5.00e-01]
 [ 1.20e+01  1.80e+02 -3.00e+00  9.50e+02  7.00e-01]
 [ 8.00e+00  2.20e+02 -7.00e+00  1.05e+03  3.00e-01]
 [ 1.10e+01  1.90e+02 -4.00e+00  9.80e+02  6.00e-01]]


## Step 2: Apply Batch Normalization

`BatchNormalization` normalizes each **feature (column)** using the mean and
variance computed **across the batch (all rows)** for that feature.

In [3]:
batch_norm_layer = tf.keras.layers.BatchNormalization()

# training=True makes it normalize using this batch's own statistics
# (mimics how it behaves during model training)
bn_output = batch_norm_layer(data, training=True)

print("Batch Normalization Output:")
print(bn_output.numpy())

Batch Normalization Output:
[[-0.16899252  0.16903019 -0.16899228  0.13736153 -0.16529465]
 [ 1.1829457  -1.1832132   1.1829455  -1.2362442   1.1570647 ]
 [-1.5209303   1.5212736  -1.52093     1.5109673  -1.4876537 ]
 [ 0.5069766  -0.5070915   0.5069766  -0.41208076  0.49588513]]


E0000 00:00:1784997199.428499     572 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## Step 3: Apply Layer Normalization

`LayerNormalization` normalizes each **sample (row)** using the mean and
variance computed **across that sample's own features**, independent of
other samples in the batch.

In [4]:
layer_norm_layer = tf.keras.layers.LayerNormalization()

ln_output = layer_norm_layer(data)

print("Layer Normalization Output:")
print(ln_output.numpy())

Layer Normalization Output:
[[-0.59690493 -0.10615659 -0.6356482   1.9601519  -0.6214423 ]
 [-0.5875966  -0.13045001 -0.62841326  1.9648051  -0.61834514]
 [-0.6052342  -0.08420092 -0.64209974  1.9556932  -0.6241585 ]
 [-0.59195447 -0.12001497 -0.6315025   1.9628465  -0.61937445]]


## Step 4: Display original input vs. both normalized outputs side by side

In [5]:
print("="*70)
print("ORIGINAL INPUT")
print("="*70)
print(data)

print("\n" + "="*70)
print("BATCH NORMALIZATION OUTPUT  (normalized per COLUMN / feature)")
print("="*70)
print(bn_output.numpy())

print("\n" + "="*70)
print("LAYER NORMALIZATION OUTPUT  (normalized per ROW / sample)")
print("="*70)
print(ln_output.numpy())

print("\n" + "="*70)
print("VERIFICATION")
print("="*70)
print("BatchNorm -> mean of each COLUMN (should be ~0):",
      np.round(bn_output.numpy().mean(axis=0), 4))
print("LayerNorm -> mean of each ROW    (should be ~0):",
      np.round(ln_output.numpy().mean(axis=1), 4))

ORIGINAL INPUT
[[ 1.00e+01  2.00e+02 -5.00e+00  1.00e+03  5.00e-01]
 [ 1.20e+01  1.80e+02 -3.00e+00  9.50e+02  7.00e-01]
 [ 8.00e+00  2.20e+02 -7.00e+00  1.05e+03  3.00e-01]
 [ 1.10e+01  1.90e+02 -4.00e+00  9.80e+02  6.00e-01]]

BATCH NORMALIZATION OUTPUT  (normalized per COLUMN / feature)
[[-0.16899252  0.16903019 -0.16899228  0.13736153 -0.16529465]
 [ 1.1829457  -1.1832132   1.1829455  -1.2362442   1.1570647 ]
 [-1.5209303   1.5212736  -1.52093     1.5109673  -1.4876537 ]
 [ 0.5069766  -0.5070915   0.5069766  -0.41208076  0.49588513]]

LAYER NORMALIZATION OUTPUT  (normalized per ROW / sample)
[[-0.59690493 -0.10615659 -0.6356482   1.9601519  -0.6214423 ]
 [-0.5875966  -0.13045001 -0.62841326  1.9648051  -0.61834514]
 [-0.6052342  -0.08420092 -0.64209974  1.9556932  -0.6241585 ]
 [-0.59195447 -0.12001497 -0.6315025   1.9628465  -0.61937445]]

VERIFICATION
BatchNorm -> mean of each COLUMN (should be ~0): [-0. -0. -0.  0.  0.]
LayerNorm -> mean of each ROW    (should be ~0): [-0.  0. -

## Step 5: Comparison

- **Batch Normalization** normalizes each feature (column) using the mean
  and variance calculated **across the whole batch** of samples, so every
  input feature ends up with roughly zero mean and unit variance for that
  batch. It's applied between layers of a network and helps training
  converge faster and more stably, since it keeps the scale of activations
  consistent as they flow through the network.
- **Layer Normalization** normalizes each sample (row) using the mean and
  variance calculated **across that sample's own features**, independent of
  any other sample. It doesn't depend on batch size at all, which makes it
  well suited to RNNs, Transformers, and situations with small or
  variable-size batches.
- **Key difference:** Batch Normalization normalizes *across the batch,
  per feature* (statistics depend on other samples), while Layer
  Normalization normalizes *across the features, per sample* (statistics
  depend only on that one sample) — this is why LayerNorm behaves the same
  whether batch size is 1 or 1000, but BatchNorm's behavior changes with
  batch size.